# Contents:

Setup:
- Download tmol wheel (colab)
- Download a few input files
- Hello World of working with tmol: create a PoseStack and score it

Fundamentals:
- Initialize the default ParameterDatabase
- Load alternate block types
- Initialize a PackedBlockTypes object from the default database
- Initialize a PackedBlockTypes object from a custom database
- Create a CanonicalOrdering from a ParameterDatabase
- Create a PackedBlockTypes object from a subset of the block types in a ParameterDatabase

Input
- Initialize a single pose PoseStack from a PDB
- Initialize a pose stack from a subset of residues in a PDB
- Initialize a single pose PoseStack from a PDB file using biotite
- Initialize a single pose PoseStack from an .mmcif file (.cif)
- Initialize a single pose PoseStack from an OpenMM set of tensors
- Initialize a single pose PoseStack with missing residues
- Load a bunch of PoseStacks from different sources and then concatenate them to a single PoseStack
- Make many copies of the same single-pose PoseStack
- Add custom residue type
- Set the scoring parameters for a new residue type (elec, etc)
- Build an extended pose from sequence
- (Ligand features??)
- Create ligand block type from .params
- Create ligand block type from .mol2
- Create ligand block type from .cif that contains ligands
- Create BiotitePoseBuildContext with RefinedResidueTypes for repeat loading of ligand PDBs
- Load in multiple ligands and then process PDBs containing those ligands

Output:
- Write a single pose PoseStack to a .pdb file
- Write a multi-pose stack out as a multi-model PDB file
- Write a multi-pose stack out to separate PDB files
- Write a single pose PoseStack to an .mmcif file
- Write a rotamer set out as a multi-model PDB
- Write ligand-containing PoseStack out to .cif file using BiotitePoseBuildContext
- Write out ligand .params file

Kinematics:
- Create an N->C fold tree for a PoseStack
- Create a simple fold tree for a multi-chain PoseStack
- Create a simple fold tree for a PoseStack with missing residues
- Create a dandelion fold tree for a PoseStack
- Create a MoveMap that enables minimization for named torsions
- Create a MoveMap that enables backbone minimization for some residues but not all
- Apply a perturbation to the rigid-body DOFs between two chains
- Assign dihedral values to all the residues in a PoseStack and calculate the coordinates

Scoring:
- Create the default score function
- Create the default score function from a custom Database
- Create the soft-rep version of the score function
- Create an empty score function
- Turn on a few terms in a score function
- Turn off a term in a score function
- Score a PoseStack
- Score a PoseStack and back-propagate through the coordinates
- Score a PoseStack and return per-residue weighted energies
- Score a PoseStack and return per-residue un-weighted energies
- Score a PoseStack and return per-residue weighted energies, weight them according to some principle, and then back-propagate the total energy.
- Add constraints to a PoseStack
- Add the same constraints to all the Poses in a PoseStack
- Add coordinate constraints to the current coordinates
- Alter the parameters for the cart-bonded energy function & rescore


Optimization
- Create a DunbrackSampler
- Add hydrogens
- Fill in side chains for a model that lacks them
- Perform fixed-sequence side-chain optimization: repack
- Repack with extra rotamers
- Add hydrogens and back-propagate
- Create a new PackerPalette subclass to handle logic of new block types
- Run minimization in double precision
- Perform cartesian minimization
- Perform kinematic minimization
- Relax a PoseStack w/ kinematic minimization
- Relax a PoseStack w/ cartesian minimization
- Relax structures generated one-at-a-time in batch format
- Idealize a structure from the PDB
- Idealize a structure from a dandelion
- Idealize just the backbone of a dandelion


In [ ]:
# For Colab
# install tmol directly from the github wheel;
!pip install https://github.com/uw-ipd/tmol/releases/download/v0.1.36/tmol-0.1.36+cu128torch2.10-cp312-cp312-linux_x86_64.whl

In [ ]:
# download some structure files so we have something to work with
!wget -O 1ubq.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1ubq.pdb
!wget -O 1s78.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1s78.pdb
!wget -O 1qys.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1qys.pdb
!wget -O 1BL8.cif https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/cif/1BL8.cif
!wget -O 3plc.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/3plc.pdb
!wget -O 10VB.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/10VB.pdb
!wget -O openfold_ubq_and_sumo.pt https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/openfold/openfold_ubq_and_sumo.pt

In [ ]:
# Make sure the files downloaded correctly; this should print the first 10 atoms
# from methionine in ubiquitin
!head 1ubq.pdb

In [ ]:
%env TMOL_USE_JIT=1

In [ ]:
import tmol

In [ ]:
print(tmol.__path__)

In [ ]:
# The hello world of working with tmol:
# Load a PDB in from disk and score it

import tmol
import torch
import os

device = torch.device("cuda", torch.cuda.current_device()) if torch.cuda.is_available() else torch.device("cpu")

# Create a pose stack from a PDB.
pose_stack = tmol.pose_stack_from_pdb('1ubq.pdb', device=device)
# A PoseStack represents a batch of molecular systems (in this case, just a single structure - ubiquitin)
# PoseStacks are optimized for compactness for efficient processing on the GPU.
# Behind the scenes, pose_stack_from_pdb uses the default ParameterDatabase;
# which currently contains the parameters necessary to treat standard
# proteins, but little else. We will see more about the ParameterDatabase later.

# Create our score function.
sfxn = tmol.beta2016_score_function(device=device)
# This tmol score function is based on the Rosetta energy function.
# The score function is composed of many terms and weights for those terms.
# In this particular case, the score function terms and weights are set to
# match the beta2016_cart score function from Rosetta3. Again, we
# are relying on the default ParameterDatabase in the background.

# Create our scoring module.
scorer = sfxn.render_whole_pose_scoring_module(pose_stack)
# The scoring module is what does the actual score evaluation.
# This is separate from the ScoreFunction because it also needs details
# about the PoseStack being scored - mainly the Residue Types being used.
# The scoring module needs those Types because each score term must assemble
# compact tensors with the data necessary to score the Residue Types that are
# in use.

# Score the PoseStack and print the output.
print(scorer(pose_stack.coords))
# Return a tensor with the score of each pose in the stack (in this case, just 1 value)

Fundamentals

In [ ]:
# - Initialize the default ParameterDatabase
default_param_db = tmol.ParameterDatabase.get_default()

In [ ]:
# - Load alternate block types
# TO DO: Kieran

In [ ]:
# - Initialize a PackedBlockTypes object from the default database
pbt = tmol.default_packed_block_types(device)

In [ ]:
# - Initialize a PackedBlockTypes object from a custom database
# TO DO

In [ ]:
# - Create a default CanonicalOrdering object
canonical_ordering = tmol.default_canonical_ordering()

In [ ]:
# - Create a CanonicalOrdering from a ParameterDatabase
# TO DO

In [ ]:
# - Create a PackedBlockTypes object from a subset of the block types in a ParameterDatabase
# TO DO

Input

In [ ]:
# - Initialize a single pose PoseStack from a PDB
pose_1ubq = tmol.pose_stack_from_pdb("1ubq.pdb", device=device)
assert pose_1ubq.max_n_blocks == 76

In [ ]:
# - Initialize a pose stack from a subset of residues in a PDB

# We have to tell the PDB reader that the first residue is not an
# N-terminus but merely is not connected to the residue that preceeds it
# and that the last residue is not a C-terminus.
res_not_connected = torch.zeros([1, 31, 2], dtype=torch.bool, device=device)
res_not_connected[0, 0, 0] = True
res_not_connected[0, 30, 1] = True

pose_1ubq_20to50 = tmol.pose_stack_from_pdb("1ubq.pdb", device=device, residue_start=20, residue_end=51, res_not_connected=res_not_connected)
assert pose_1ubq_20to50.inter_residue_connections[0,  0, 0, 0] == -1
assert pose_1ubq_20to50.inter_residue_connections[0, 30, 1, 0] == -1


In [ ]:
# - Initialize a single pose PoseStack from a PDB file using biotite
import biotite.structure
import biotite.structure.io.pdb 

# TEMP
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite

bt_pdb_file = biotite.structure.io.pdb.PDBFile.read("1ubq.pdb")
bt_struct = bt_pdb_file.get_structure()
if isinstance(bt_struct, biotite.structure.AtomArrayStack):
    bt_struct = bt_struct[0]

pose_1ubq_bt = pose_stack_from_biotite(bt_struct, device)

In [ ]:
# - Initialize a single pose PoseStack from an mmcif file (.cif)
import biotite.structure
from biotite.structure.io.pdbx import CIFFile, set_structure

# TEMP
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite

bt_struct = biotite.structure.io.load_structure(
    "1BL8.cif", extra_fields=["occupancy", "b_factor"]
)
if isinstance(bt_struct, biotite.structure.AtomArrayStack):
    bt_struct = bt_struct[0]

# TEMP: add pose_stack_from_biotite to API and modify this line
pose_1bl8 = pose_stack_from_biotite(bt_struct, device)

In [ ]:
# - Initialize a single pose PoseStack from an OpenFold set of tensors

# here is a dictionary with the set of tensors that OpenFold produces when
# asked to predict the structures of ubiquitin and sumo; in particular, tmol
# reads from the "aatype", "positions" and "chain_index" tensors.

openfold_ubq_and_sumo_pred = torch.load("openfold_ubq_and_sumo.pt", map_location=device)

ps_ubq_sumo = tmol.pose_stack_from_openfold(openfold_ubq_and_sumo_pred)
assert len(ps_ubq_sumo) == 2
assert ps_ubq_sumo.max_n_blocks == openfold_ubq_and_sumo_pred["positions"].shape[2]
assert ps_ubq_sumo.coords.device == device


In [ ]:
# - Initialize a single pose PoseStack with missing residues
# This particular structure has a few regions where the backbone is missing
pose_1s78 = tmol.pose_stack_from_pdb("1s78.pdb", device=device)
# TO DO: Assert that some residues are not bound to their upper / lower conns

In [ ]:
# - Load a bunch of PoseStacks from different sources and then concatenate them to a single PoseStack
ps_3plc = tmol.pose_stack_from_pdb("3plc.pdb", device=device)
ps_1qys = tmol.pose_stack_from_pdb("1qys.pdb", device=device)

# TEMP: Add PoseStackBuilder to API
from tmol.pose.pose_stack_builder import PoseStackBuilder
pose_stack_3 = PoseStackBuilder.from_poses([pose_1ubq, ps_3plc, ps_1qys], device=device)
assert pose_stack_3.n_poses == 3

In [ ]:
# - Make many copies of the same single-pose PoseStack
# You can create a list of shallow copies of a single-pose PoseStack
# and the PoseStackBuilder will expand them into complete and fully
# independent poses.
ten_1ubqs = PoseStackBuilder.from_poses([pose_1ubq] * 10, device=device)
assert ten_1ubqs.n_poses == 10
assert ten_1ubqs.coords.shape[0] == 10

In [ ]:
# - Add custom residue type
# TO DO


In [ ]:
# - Set the scoring parameters for a new residue type (elec, etc)
# TO DO


In [ ]:
# - Build an extended pose from sequence
# TO DO

Output

In [ ]:
# - Write a single pose PoseStack to a .pdb file
tmol.write_pose_stack_pdb(pose_1ubq, "1ubq_out.pdb")
assert os.path.isfile("1ubq_out.pdb")

In [ ]:
# - Write a multi-pose stack out as a multi-model PDB file
tmol.write_pose_stack_pdb(ten_1ubqs, "ten_1ubqs.pdb")
assert os.path.isfile("ten_1ubqs.pdb")
def nlines_from_file(fname):
  with open(fname) as fid:
    lines = fid.readlines()
  return len(lines)
nlines_1ubq = nlines_from_file("1ubq.pdb")
nlines_ten_1ubqs = nlines_from_file("ten_1ubqs.pdb")
assert 10 * nlines_1ubq <= nlines_ten_1ubqs

In [ ]:
# - Write a multi-pose stack out to separate PDB files

# take advantage of PoseStack's split() method to create a single-pose PoseStack
for i in range(10):
  pose_i = ten_1ubqs.split(i)
  tmol.write_pose_stack_pdb(pose_i, f"1ubq_{i:04}.pdb")

for i in range(10):
  assert os.path.isfile(f"1ubq_{i:04}.pdb")

In [ ]:
# - Write a single pose PoseStack to an .mmcif file
# TO DO


In [ ]:
# - Write a rotamer set out as a multi-model PDB
# TO DO


Kinematics

In [ ]:
# - Create an N->C fold tree (a fold *forest* in tmol) for a PoseStack
import numpy

# Option 1: reasonable_fold_forest() produces a simple fold tree for
# each pose in the PoseStack of one N->C edge per chain.
ff1_1ubq = tmol.FoldForest.reasonable_fold_forest(pose_1ubq)

# Option 2: specify the fold forest explicitly. 
# Each edge is a 3-tuple of (edge_type, start_residue, end_residue)
#
# tmol includes a new kind of jump edge: a root-jump. This connects
# the downstream residue to the root of the fold forest. Each 
# fold tree must include at least one root-jump edge, but may
# include arbitrarily many. The start_residue of a root-jump edge
# is always the sentinel value of -1.
# 
# For this N->C fold tree, we define a root-jump to residue 0, and a polymer
# edge from 0 to the last residue in the pose.
# 
# The edges array should be [n_poses, max_n_edges, 3] with the sentinel value
# of -1 used for unused edges.
edges = numpy.full((1, 2, 3), -1, dtype=int)
edges[0, 0] = [tmol.EdgeType.root_jump, -1, 0]
edges[0, 1] = [tmol.EdgeType.polymer, 0, 75]
ff2_1ubq = tmol.FoldForest.from_edges(edges)

In [ ]:
# - Create a set of N->C fold trees for a multi-pose PoseStack of single-chain PDBs
ff1_ps3 = tmol.FoldForest.reasonable_fold_forest(pose_stack_3)

# Option 2: build the edges array explicitly, as above.
edges = numpy.full((3, 2, 3), -1, dtype=int)
edges[:, 0] = numpy.array([tmol.EdgeType.root_jump, -1, 0], dtype=int)[None, :]
n_res = pose_stack_3.n_res_per_pose.cpu().numpy()
edges[:, 1, 0] = tmol.EdgeType.polymer
edges[:, 1, 1] = 0
edges[:, 1, 2] = n_res - 1
ff2_ps3 = tmol.FoldForest.from_edges(edges)

In [ ]:
# - Create a simple fold tree for a multi-chain PoseStack
ps_10vb = tmol.pose_stack_from_pdb("10VB.pdb", device=device)
ff_10vb = tmol.FoldForest.reasonable_fold_forest(ps_10vb)


In [ ]:
# - Create a simple fold tree for a PoseStack with missing residues

# TO DO: Fix logic for reasonable_fold_forest so that it inserts jumps
# between blocks that are in the same chain but that are not connected.
ff_1s78 = tmol.FoldForest.reasonable_fold_forest(pose_1s78)

In [ ]:
# - Create a dandelion fold tree for a PoseStack

# Many NNs, such as OpenFold, produce structures as one coordinate frame
# per residue and dihedrals for the side chains. Kinematically,
# we can reproduce that by describing a tree with one root-jump
# for every residue -- the side chains will still be described
# with internal geometries. This system is like a dandelion with 
# tons of tiny stalks holding tiny seeds at the end all connected
# to a central hub. With such a kin_forest, e.g., it is possible to minimize
# the system in the same set of DOFs that the NN has access to.

#
edges = numpy.full((3, pose_stack_3.max_n_blocks, 3), -1, dtype=int)
edges[:, :, 0] = tmol.EdgeType.root_jump
edges[:, :, 2] = numpy.arange(pose_stack_3.max_n_blocks, dtype=int)[None, :]

# sentinel out the edges that are out-of-bounds for each pose
is_unreal_res = (pose_stack_3.block_type_ind == -1).cpu().numpy()
edges[is_unreal_res, :] = -1

dandelion_ff = tmol.FoldForest.from_edges(edges)

In [ ]:
# - Create a MoveMap that enables minimization for named torsions
mm_1ubq_all = tmol.MoveMap.from_pose_stack(pose_1ubq)
mm_1ubq_all.move_all_named_torsions = True

In [ ]:
# - Create a MoveMap that enables backbone minimization for some residues but not all

# Let's turn on named-torsion minimization for residues 10-39
mm_1ubq_some = tmol.MoveMap.from_pose_stack(pose_1ubq)
mm_1ubq_some.set_move_all_named_torsions_for_blocks(0, torch.arange(30, dtype=torch.int64, device=device) + 10)

In [ ]:
# - Apply a perturbation to the rigid-body DOFs between two chains
# TO DO

In [ ]:
# - Assign dihedral values to all the residues in a PoseStack and calculate the coordinates
# TO DO

Scoring

In [ ]:
# - Create the default score function
sfxn = tmol.beta2016_score_function(device)

In [ ]:
# - Create the default score function from a custom Database
# TO DO

In [ ]:
# - Create the soft-rep version of the score function
# TO DO

In [ ]:
# - Create an empty score function
sfxn_empty = tmol.ScoreFunction(default_param_db, device)

In [ ]:
# - Turn on a few terms in a score function
sfxn_rep_and_hbonds = tmol.ScoreFunction(default_param_db, device)
sfxn_rep_and_hbonds.set_weight(tmol.ScoreType.fa_ljrep, 0.55)
sfxn_rep_and_hbonds.set_weight(tmol.ScoreType.hbond, 1.0)

In [ ]:
# - Turn off a term in a score function
# TO DO; fix bug
# print(len(sfxn_rep_and_hbonds.all_terms()))

sfxn_rep_and_hbonds.set_weight(tmol.ScoreType.hbond, 0)
assert len(sfxn_rep_and_hbonds.all_terms()) == 1

sfxn_rep_and_hbonds.set_weight(tmol.ScoreType.fa_ljrep, 0)
assert len(sfxn_rep_and_hbonds.all_terms()) == 0


In [ ]:
# - Score a PoseStack
wpsm = sfxn.render_whole_pose_scoring_module(pose_1ubq)
score = wpsm(pose_1ubq.coords)

In [ ]:
# - Score a PoseStack and back-propagate through the coordinates
score2 = wpsm(pose_1ubq.coords)
score2.sum().backward()

In [ ]:
# - Score a PoseStack and return per-residue weighted energies
bpsm = sfxn.render_block_pair_scoring_module(pose_1ubq)
block_pair_scores = bpsm(pose_1ubq.coords)
assert block_pair_scores.shape == (1, pose_1ubq.max_n_blocks, pose_1ubq.max_n_blocks)

In [ ]:
# - Score a PoseStack and return per-residue un-weighted energies
unweighted_block_pair_scores = bpsm.unweighted_scores(pose_1ubq.coords)
assert unweighted_block_pair_scores.shape == (len(sfxn.all_score_types()), 1, pose_1ubq.max_n_blocks, pose_1ubq.max_n_blocks)

In [ ]:
# - Score a PoseStack and return per-residue weighted energies, weight them according to some principle,
# and then back-propagate the total reweighted energy.
pose_1s78.coords.requires_grad_()
bpsm_1s78 = sfxn.render_block_pair_scoring_module(pose_1s78)
bps = bpsm_1s78(pose_1s78.coords)
tot_reg = bps.sum()

# in 1s78, there are 555 residues in the antigen; the remaining residues are in the antibody
upweight_interchain_intxns = torch.ones((1, 991, 991), dtype=float, device=device)
# energies are written to the upper-triangle of the n-res x n-res table
upweight_interchain_intxns[0, 0:555, 555:991] = 2.0

tot_upweight_iface = (bps * upweight_interchain_intxns).sum()
print("tot reg", tot_reg, "tot_upweight_iface", tot_upweight_iface)
dcoords = tot_upweight_iface.backward()

In [ ]:
# - Add constraints to a PoseStack

# The PoseStack carries a ConstraintSet object that may be shared between
# multiple PoseStacks; thus ConstraintSet is immutable. Instead of
# being able to modify a ConstraintSet, the class makes it easy to
# create a new ConstraintSet with the contents you want.
# PoseStack is the same way: you cannot modify the ConstraintSet in 
# an existing PoseStack, but using attr.evolve(...) you can readily
# construct a new PoseStack that holds your newly constructed 
# ConstraintSet
import attr

assert pose_1ubq.constraint_set is None
cst_set = tmol.ConstraintSet.create_empty(device=device, n_poses=pose_1ubq.n_poses)

# let's create a constraint set with CA distances
# between all pairs of residues
start_ca_coords = []
res_inds = []
atom_inds = []
coords = pose_1ubq.coords.cpu()
for i in range(pose_1ubq.max_n_blocks):
    i_bt = pose_1ubq.block_type(0, i)
    # In general, not all block types will have a "CA" atom
    # but in the case of 1ubq, they do happen to.
    if "CA" in i_bt.atom_names_set:
        res_inds.append(i)
        at_ind = i_bt.atom_to_idx["CA"]
        atom_inds.append(at_ind)
        start_ca_coords.append(coords[0, pose_1ubq.block_coord_offset64[0, i] + at_ind])

start_ca_coords = torch.stack(start_ca_coords).to(device=device)
print("start_ca_coords", start_ca_coords.shape)
res_inds = torch.tensor(res_inds, dtype=torch.int64, device=device)
atom_inds = torch.tensor(atom_inds, dtype=torch.int64, device=device)

ca_dists = torch.linalg.norm(start_ca_coords[None, :, :] - start_ca_coords[:, None, :], dim=2)
print("ca_dists", ca_dists.shape)
n_res_arange = torch.arange(len(start_ca_coords), dtype=torch.int64)
is_upper_triangle = n_res_arange[:, None] < n_res_arange[None, :]
nz_upper_triangle_r1, nz_upper_triangle_r2 = torch.nonzero(is_upper_triangle, as_tuple=True)

n_csts = nz_upper_triangle_r1.shape[0]
cst_atoms = torch.zeros((n_csts, 2, 3), dtype=torch.int64, device=device)
cst_params = torch.zeros((n_csts, 4), dtype=torch.float32, device=device)
cst_atoms[:, :, 0] = 0 # pose index
cst_atoms[:, 0, 1] = res_inds[nz_upper_triangle_r1] # atom1 residue index
cst_atoms[:, 1, 1] = res_inds[nz_upper_triangle_r2] # atom2 residue index
cst_atoms[:, 0, 2] = atom_inds[nz_upper_triangle_r1] # atom1 atom index within its residue
cst_atoms[:, 1, 2] = atom_inds[nz_upper_triangle_r2] # atom2 atom index within its residue

cst_params[:, 0] = ca_dists[nz_upper_triangle_r1, nz_upper_triangle_r2]
cst_params[:, 1] = 0.5 # 0.5A standard deviation

cst_set = cst_set.add_constraints(
    tmol.ConstraintEnergyTerm.harmonic,
    cst_atoms,
    cst_params
)
pose_1ubq_w_csts = attr.evolve(pose_1ubq, constraint_set=cst_set)


In [ ]:
wpsm_w_csts = sfxn.render_whole_pose_scoring_module(pose_1ubq_w_csts)
score = wpsm_w_csts(pose_1ubq_w_csts.coords)
print("score w/ constraints", score)

In [ ]:
# - Add the same constraints to all the Poses in a PoseStack
cst_set_10 = tmol.ConstraintSet.create_empty(device=device, n_poses=ten_1ubqs.n_poses)
cst_set_10 = cst_set_10.add_constraints_to_all_poses(tmol.ConstraintEnergyTerm.harmonic, cst_atoms, cst_params)
ten_1ubqs_w_csts = attr.evolve(ten_1ubqs, constraint_set=cst_set_10)

In [ ]:
# - Add coordinate constraints to the current coordinates

# Option 1: use the utility function
# TEMP!
pose_1ubq_w_ca_coord_csts1 = tmol.constrain_all_ca(pose_1ubq)

# Option 2: use the other utility function
pose_1ubq_w_ca_coord_csts2 = tmol.create_mainchain_coordinate_constraints(pose_1ubq)

In [ ]:
# - Alter the parameters for the cart-bonded energy function & rescore

# The same immutable + shallow-copy + "evolve" strategy is how scoring
# parameters are controlled. In this case, the CartBondedDatabase
# keeps a hash of all of its parameters in order to ensure it uses
# the appropriate set of tensors, and so it provides its own 
# evolve-like method for construction from a dictionary of parameters:
# CartBondedDatabase.from_cartres_dict.
#
# This is a somewhat advanced feature, so CartBondedDatabase is not imported
# with tmol by default, so we have to give its full scope

# Let's imagine that we want to adjust the strength of only the 
# bond distance between PRO CD and backbone N to make it stronger.
import copy

cart_db = default_param_db.scoring.cartbonded

# replace the default spring constant on the peptide bond with a stronger one
pro_params = cart_db.residue_params["PRO"]
pro_length_params = pro_params.length_parameters
ind, cd_n_bond_params = next((i, p) for (i, p) in enumerate(pro_length_params) if p.atm1 == "N" and p.atm2 == "CD")
alt_cd_n_bond_params = attr.evolve(cd_n_bond_params, K=244)  # double the default strength of ~122 kcal/mol*A
alt_length_params = pro_length_params[0:ind] + (alt_cd_n_bond_params,) + pro_length_params[ind+1:]
alt_pro_params = attr.evolve(pro_params, length_parameters=alt_length_params)
alt_residue_params = copy.deepcopy(cart_db.residue_params)
alt_residue_params["PRO"] = alt_pro_params

alt_cart_db = tmol.database.scoring.CartBondedDatabase.from_cartres_dict(cartres_dict=alt_residue_params)

alt_score_db = attr.evolve(default_param_db.scoring, cartbonded=alt_cart_db)
alt_param_db = attr.evolve(default_param_db, scoring=alt_score_db)

sfxn_alt = tmol.beta2016_score_function(device, param_db=alt_param_db)

wpsm_std = sfxn.render_whole_pose_scoring_module(pose_1ubq)
wpsm_alt = sfxn_alt.render_whole_pose_scoring_module(pose_1ubq)
score_std = wpsm_std(pose_1ubq.coords)
score_alt = wpsm_alt(pose_1ubq.coords)
assert score_std < score_alt

Optimization

In [ ]:
# - Create a DunbrackSampler

# constructing this object is a relatively heavy weight operation, so if you
# will be invoking the packer repeatedly, it's worthwhile to construct it
# once and then hold on to it
dun_sampler = tmol.create_dunbrack_sampler_from_database(default_param_db, device)

In [ ]:
# - Perform fixed-sequence side-chain optimization: repack

# we will invoke "the packer": the module in tmol that optimizes the
# discrete side-chain conformation assignment.
# Steps:
# 1. Create a PackerPalette
# 2. Create a PackerTask using the palette
# 3. Configure the PackerTask to say "optimize the sequence, but don't change it"
# 4. Invoke PackRotamers

# 1.
# The default PackerPalette controls the initialization of the PackerTask
# to allow design from LCAAs to other LCAAs, DCAAs to other DCAAs, and
# otherwise only allows the original block types. The PackerPalette class is
# meant to be subclassed, so if you have more complex logic for which
# block types to consider at any given position, then you should write
# your own PackerPalette subclass.
palette = tmol.PackerPalette()

# 2.
task = tmol.PackerTask(pose_1ubq, palette)

# 3.
task.restrict_to_repacking()
task.add_conformer_sampler(dun_sampler)
task.add_conformer_sampler(tmol.FixedAAChiSampler())
# Beware: the native rotamer is often better than the naive rotamers
# and so if you add the IncludeCurrentSampler in e.g. a sequence
# recovery benchmark, you will be biasing the energies of the native
# sequence. For most modeling problems, though, you really do want
# to keep the input rotamer.
task.add_conformer_sampler(tmol.IncludeCurrentSampler())

# 4.
pose_1ubq_repacked = tmol.pack_rotamers(pose_1ubq, sfxn, task, verbose=True)


In [ ]:
wpsm = sfxn.render_whole_pose_scoring_module(pose_1ubq_repacked)
print("score", wpsm(pose_1ubq_repacked.coords))

In [ ]:
# - Add hydrogens

# Most NNs model only the heavy atoms; tmol's energy function requires Hs.
# The basic pathway of creating a PoseStack will place alaphatic hydrogens
# in their ideal geometries, however, the hydroxyl hydrogens will not be
# optimized. To optimize Hs, you can use the OptHSampler when invoking
# packer; only repack the hydroxyl positions

optH_task = tmol.PackerTask(pose_1ubq, palette)
optH_task.restrict_to_repacking()
optH_task.add_conformer_sampler(tmol.IncludeCurrentSampler())
optH_task.add_conformer_sampler(tmol.OptHSampler())

pose_1ubq_optH_repacked = tmol.pack_rotamers(pose_1ubq, sfxn, task, verbose=True)

In [ ]:
# - Fill in side chains for a model that lacks them
# TO DO
# Sometimes your models will only include the backbone, or
# you are reading from the PDB where the electron density
# was absent for a side chain and the crystalographer did
# not add its conformation.

# When working with a structure file, the
# pose_stack_from_biotite pathway will automatically 
# build in missing side chains.



In [ ]:
# - Repack with extra rotamers

# the "extra chi" flags in the PackerTask will trigger
# sampling at +/- 1stdev
palette = tmol.PackerPalette()
task = tmol.PackerTask(pose_1ubq, palette)

# 3.
task.restrict_to_repacking()
task.add_conformer_sampler(dun_sampler)
task.add_conformer_sampler(tmol.FixedAAChiSampler())
task.add_conformer_sampler(tmol.IncludeCurrentSampler())
task.or_expand_chi(1)
task.or_expand_chi(2)

# 4.
pose_1ubq_repacked_ex1ex2 = tmol.pack_rotamers(pose_1ubq, sfxn, task, verbose=True)

In [ ]:
wpsm = sfxn.render_whole_pose_scoring_module(pose_1ubq_repacked_ex1ex2)
print("score", wpsm(pose_1ubq_repacked_ex1ex2.coords))

In [ ]:
# - Add hydrogens and back-propagate

# The addition of missing alaphatic hydrogen atoms is differentiable
# and so it is possible to start from a model output from a NN
# which does not have hydrogens, build those hydrogens, score the
# structure, and then have the forces applied on those hydrogens
# backprop to the heavy-atoms they came from. All of that comes
# for free using the pose_stack_from_openfold utility.

openfold_ubq_and_sumo_pred2 = torch.load("openfold_ubq_and_sumo.pt", map_location=device)
# pretend an NN just gave us this
openfold_ubq_and_sumo_pred2["positions"].requires_grad_()
ps_ubq_sumo2 = tmol.pose_stack_from_openfold(openfold_ubq_and_sumo_pred2)
wpsm = sfxn.render_whole_pose_scoring_module(ps_ubq_sumo2)
score = wpsm(ps_ubq_sumo2.coords)
score.sum().backward()

In [ ]:
# - Create a new PackerPalette subclass to handle logic of new block types
# TO DO

In [ ]:
# - Run minimization in double precision
# TO DO

In [ ]:
# - Perform cartesian minimization

cart_minimized_pose_1ubq = tmol.run_cart_min(pose_1ubq, sfxn)


In [ ]:
# - Perform kinematic minimization

# The run_kin_min function will perform minimization in
# internal degrees of freedom.

# from the kinematics section above
ff1_1ubq = tmol.FoldForest.reasonable_fold_forest(pose_1ubq)
mm_1ubq_all = tmol.MoveMap.from_pose_stack(pose_1ubq)
mm_1ubq_all.move_all_named_torsions = True

wpsm = sfxn.render_whole_pose_scoring_module(pose_1ubq)
score_before = wpsm(pose_1ubq.coords)

minimized_pose_1ubq = tmol.run_kin_min(pose_1ubq, sfxn, ff1_1ubq, mm_1ubq_all)

score_after = wpsm(minimized_pose_1ubq.coords)
assert score_after < score_before

In [ ]:
# - Relax a PoseStack w/ kinematic minimization

packer_palette = tmol.PackerPalette()

# from the kinematics section above
ff1_1ubq = tmol.FoldForest.reasonable_fold_forest(pose_1ubq)
mm_1ubq_all = tmol.MoveMap.from_pose_stack(pose_1ubq)
mm_1ubq_all.move_all_named_torsions = True

kin_relaxed_1ubq = tmol.kin_fast_relax(pose_1ubq, sfxn, packer_palette, mm_1ubq_all, ff1_1ubq, verbose=True)

In [ ]:
# - Relax a PoseStack w/ cartesian minimization
packer_palette = tmol.PackerPalette()

# default CartesianMoveMap allows all atoms to move
cart_mm = tmol.CartesianMoveMap()

cart_relaxed_1ubq = tmol.cartesian_fast_relax(pose_1ubq, sfxn, packer_palette, cart_mm, verbose=True)

In [ ]:
# - Relax structures generated one-at-a-time in batch format

# Though the two relax cells above were a) on single-pose PoseStacks, and
# b) on small structures, where the GPU really sings is with large
# structures and with many-pose PoseStacks. Then, 


In [ ]:
# - Idealize a structure from the PDB
# TO DO

In [ ]:
# - Idealize a structure from a dandelion
# TO DO

In [ ]:
# - Idealize just the backbone of a dandelion
# TO DO